In [ ]:
import torch
from transformers import MobileBertTokenizer, MobileBertForSequenceClassification


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = MobileBertTokenizer.from_pretrained('google/mobilebert-uncased')
model = MobileBertForSequenceClassification.from_pretrained('cssupport/mobilebert-sql-injection-detect')
model.to(device)
model.eval()

def predict(text:str, threshold:float = 0.7) -> bool:
    inputs = tokenizer(text, padding=False, truncation=True, return_tensors='pt', max_length=512)
    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=1)
    predicted_class = torch.argmax(probabilities, dim=1).item()
    confidence = probabilities[0][predicted_class].item()
    
    if predicted_class > threshold:
        return True
    return False

In [ ]:
my_df = my_df[["Query", "Label"]]
my_df

In [ ]:
my_df["prediction"] = [1 if predict(row.Query) else 0 for row in my_df.itertuples()]

In [ ]:
import numpy as np

sum(my_df['Label'] == my_df['prediction'])/len(my_df)

In [ ]:
import pandas as pd

path = "data/sql_dataset/Modified_SQL_Dataset.csv"
df = pd.read_csv(path)
my_df = df[::30][:1000]
sum(my_df['Label'])/len(my_df)
my_df